# Notebook 03 — Signal Detection

Applies disproportionality analysis on the serious-event subset.

Outputs:
- `outputs/signals/flagged_signals.csv` — PRR + ROR + CI + priority
- `outputs/signals/timeseries_signals.csv` — quarterly case counts for top signals
- `outputs/signals/outcome_model_coefs.csv` — logistic regression odds ratios

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

Path('../outputs/signals').mkdir(parents=True, exist_ok=True)

df = pd.read_csv('../data/faers_serious.csv', usecols=['drugname', 'reaction', 'outc_cod', 'report_date', 'is_biologic'])
df['report_date'] = pd.to_datetime(df['report_date'], errors='coerce')

print(f'Serious reports: {len(df):,}')
print(f'Unique drugs:    {df["drugname"].nunique():,}')
print(f'Unique reactions:{df["reaction"].nunique():,}')

Serious reports: 224,843
Unique drugs:    6,995
Unique reactions:8,044


## 1. Build contingency table using marginal totals (vectorised)

Vectorised over all pairs simultaneously — avoids the slow row-by-row loop
in the original `compute_prr` function.

In [2]:
# Count observed drug-reaction pairs
pair_counts = (
    df.groupby(['drugname', 'reaction'])
      .size()
      .rename('a')
      .reset_index()
)

# Keep pairs with at least 3 cases
pair_counts = pair_counts[pair_counts['a'] >= 3].copy()

# Marginal totals
drug_totals     = df['drugname'].value_counts()
reaction_totals = df['reaction'].value_counts()
N = len(df)

pair_counts['drug_total']     = pair_counts['drugname'].map(drug_totals)
pair_counts['reaction_total'] = pair_counts['reaction'].map(reaction_totals)

# 2×2 contingency cells
pair_counts['b'] = pair_counts['drug_total']     - pair_counts['a']
pair_counts['c'] = pair_counts['reaction_total'] - pair_counts['a']
pair_counts['d'] = N - pair_counts['a'] - pair_counts['b'] - pair_counts['c']

# Filter invalid cells
valid = (pair_counts[['a', 'b', 'c', 'd']] > 0).all(axis=1)
calc  = pair_counts[valid].copy()

print(f'Candidate pairs (a>=3, all cells >0): {len(calc):,}')

Candidate pairs (a>=3, all cells >0): 13,383


## 2. Compute PRR and ROR with 95 % CI

**Added vs original:** ROR is computed alongside PRR.
ROR is preferred when the background rate is high; PRR is more conservative.
Regulators expect both for a credible safety signal report.

In [3]:
# PRR = (a/(a+b)) / (c/(c+d))
calc['prr'] = (
    (calc['a'] / (calc['a'] + calc['b'])) /
    (calc['c'] / (calc['c'] + calc['d']))
)

# PRR 95 % CI via delta method on log scale
calc['prr_se'] = np.sqrt(
    (1/calc['a']) - (1/(calc['a']+calc['b'])) +
    (1/calc['c']) - (1/(calc['c']+calc['d']))
)
calc['prr_lower_ci'] = np.exp(np.log(calc['prr']) - 1.96 * calc['prr_se'])
calc['prr_upper_ci'] = np.exp(np.log(calc['prr']) + 1.96 * calc['prr_se'])

# ROR = (a*d) / (b*c)
calc['ror'] = (calc['a'] * calc['d']) / (calc['b'] * calc['c'])

# ROR 95 % CI via delta method on log scale
calc['ror_se'] = np.sqrt(
    (1/calc['a']) + (1/calc['b']) + (1/calc['c']) + (1/calc['d'])
)
calc['ror_lower_ci'] = np.exp(np.log(calc['ror']) - 1.96 * calc['ror_se'])
calc['ror_upper_ci'] = np.exp(np.log(calc['ror']) + 1.96 * calc['ror_se'])

# Flag signals: PRR > 2 AND lower 95% CI > 1
signals = calc[
    (calc['prr'] > 2) & (calc['prr_lower_ci'] > 1)
].copy()

# Round
for col in ['prr', 'prr_lower_ci', 'prr_upper_ci', 'ror', 'ror_lower_ci', 'ror_upper_ci']:
    signals[col] = signals[col].round(2)

signals = signals.rename(columns={'drugname': 'drug', 'a': 'cases'})

print(f'Flagged signals: {len(signals):,}')

Flagged signals: 9,205


## 3. Priority classification

In [4]:
def classify(row):
    if row['prr'] > 3 and row['cases'] >= 10:
        return 'High'
    elif row['prr'] > 2:
        return 'Medium'
    return 'Low'

signals['priority'] = signals.apply(classify, axis=1)

print(signals['priority'].value_counts())
print('\nTop 10 signals by PRR:')
print(
    signals[['drug','reaction','cases','prr','ror','priority']]
    .sort_values('prr', ascending=False)
    .head(10)
    .to_string(index=False)
)

priority
Medium    7745
High      1459
Low          1
Name: count, dtype: int64

Top 10 signals by PRR:
                                                                drug                        reaction  cases       prr       ror priority
                                                  MONTELUKAST SODIUM     NEUROPSYCHOLOGICAL SYMPTOMS     93 104984.38 197092.44     High
                                                            IDELVION            FACTOR IX INHIBITION      3  67449.90  96356.57   Medium
                                           BUPIVACAINE HYDROCHLORIDE FOETAL EXPOSURE DURING DELIVERY     23  48775.19  86168.73     High
                                         ABACAVIR SULFATE\LAMIVUDINE       FOETAL GROWTH ABNORMALITY      4  47331.37  59952.80   Medium
                                                         DINUTUXIMAB         NEUROBLASTOMA RECURRENT     21  42530.86  98351.31     High
                                  SUMATRIPTAN; SUMATRIPTAN SUCCINATE      

## 4. Save flagged signals

In [5]:
out_cols = ['drug', 'reaction', 'cases',
            'prr', 'prr_lower_ci', 'prr_upper_ci',
            'ror', 'ror_lower_ci', 'ror_upper_ci',
            'priority']

signals[out_cols].sort_values('prr', ascending=False).to_csv(
    '../outputs/signals/flagged_signals.csv', index=False
)
print('Saved: outputs/signals/flagged_signals.csv')

Saved: outputs/signals/flagged_signals.csv


## 5. Time-series — quarterly case counts for top-50 high-priority signals

**New in v2:** shows how reporting evolved over the drug's post-market lifecycle.
Useful for demonstrating signal emergence after a label update or safety communication.

In [6]:
if df['report_date'].notna().any():
    df_ts = pd.read_csv('../data/faers_serious.csv',
                        usecols=['drugname', 'reaction', 'report_date'])
    df_ts['report_date'] = pd.to_datetime(df_ts['report_date'], errors='coerce')
    df_ts = df_ts.dropna(subset=['report_date'])
    df_ts['quarter'] = df_ts['report_date'].dt.to_period('Q').astype(str)

    top_signals = (
        signals[signals['priority'] == 'High']
        .head(50)[['drug', 'reaction']]
        .rename(columns={'drug': 'drugname'})
    )

    ts_records = []
    for _, row in top_signals.iterrows():
        subset = df_ts[
            (df_ts['drugname'] == row['drugname']) &
            (df_ts['reaction'] == row['reaction'])
        ]
        ts = subset.groupby('quarter').size().reset_index(name='cases')
        ts['drug']     = row['drugname']
        ts['reaction'] = row['reaction']
        ts_records.append(ts)

    if ts_records:
        ts_df = pd.concat(ts_records, ignore_index=True)
        ts_df.to_csv('../outputs/signals/timeseries_signals.csv', index=False)
        print(f'Time-series saved: {len(ts_df):,} rows across {len(top_signals)} signals')
else:
    print('report_date column missing — skipping time-series.')

Time-series saved: 626 rows across 50 signals


## 6. Outcome prediction model (logistic regression)

**New in v2:** predicts serious vs non-serious outcome from drug class, sex,
age group, top drugs and reactions. Outputs odds ratios for dashboard display.

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# LOAD DATA
# =========================================================

model_df = pd.read_csv(
    '../data/faers_clean.csv',
    low_memory=False
)

print("Dataset Shape:", model_df.shape)

# =========================================================
# CREATE TARGET VARIABLE
# =========================================================

# Serious outcomes:
# DE = Death
# HO = Hospitalization
# LT = Life-threatening
# DS = Disability

model_df['target'] = model_df['outc_cod'].isin(
    ['DE', 'HO', 'LT', 'DS']
).astype(int)

# =========================================================
# ENCODE BASIC FEATURES
# =========================================================

# Encode patient sex
sex_encoder = LabelEncoder()

model_df['sex_enc'] = sex_encoder.fit_transform(
    model_df['patient_sex']
    .fillna('UNKNOWN')
    .astype(str)
)

# Encode age group
age_encoder = LabelEncoder()

model_df['age_enc'] = age_encoder.fit_transform(
    model_df['age_group']
    .fillna('Unknown')
    .astype(str)
)

# Biologic flag
if 'is_biologic' in model_df.columns:
    model_df['bio_enc'] = (
        model_df['is_biologic']
        .fillna(0)
        .astype(int)
    )
else:
    model_df['bio_enc'] = 0

# =========================================================
# CREATE TOP DRUG FEATURES
# =========================================================

top_drugs = (
    model_df['drugname']
    .fillna('UNKNOWN')
    .value_counts()
    .head(20)
    .index
)

print("\nTop Drugs Used:")
print(top_drugs.tolist())

for drug in top_drugs:

    safe_name = (
        str(drug)
        .replace(' ', '_')
        .replace('-', '_')
        .replace('/', '_')
        .replace('(', '')
        .replace(')', '')
    )

    model_df[f'drug_{safe_name}'] = (
        model_df['drugname'] == drug
    ).astype(np.int8)

# =========================================================
# CREATE TOP REACTION FEATURES
# =========================================================

top_rxns = (
    model_df['reaction']
    .fillna('UNKNOWN')
    .value_counts()
    .head(20)
    .index
)

print("\nTop Reactions:")
print(top_rxns.tolist())

for rxn in top_rxns:

    safe_name = (
        str(rxn)
        .replace(' ', '_')
        .replace('-', '_')
        .replace('/', '_')
        .replace('(', '')
        .replace(')', '')
    )

    model_df[f'rxn_{safe_name}'] = (
        model_df['reaction'] == rxn
    ).astype(np.int8)

# =========================================================
# SELECT FEATURES
# =========================================================

drug_cols = [
    c for c in model_df.columns
    if c.startswith('drug_') and c != 'drugname'
]

rxn_cols = [
    c for c in model_df.columns
    if c.startswith('rxn_') and c != 'reaction'
]

feat_cols = [
    'sex_enc',
    'age_enc',
    'bio_enc'
] + drug_cols + rxn_cols

print("\nNumber of Features:", len(feat_cols))

# =========================================================
# BUILD FEATURE MATRIX
# =========================================================

X = model_df[feat_cols].copy()

# Convert everything safely to numeric
X = X.apply(pd.to_numeric, errors='coerce')

# Replace missing values
X = X.fillna(0)

# Reduce memory usage
X = X.astype(np.float32)

# Target variable
y = model_df['target']

print("\nFeature Matrix Shape:", X.shape)

# =========================================================
# TRAIN LOGISTIC REGRESSION MODEL
# =========================================================

print("\nTraining Logistic Regression Model...")

model = LogisticRegression(
    max_iter=200,
    solver='liblinear',
    random_state=42
)

model.fit(X, y)

print("Model Training Complete!")

# =========================================================
# EXTRACT COEFFICIENTS
# =========================================================

coef_df = pd.DataFrame({
    'feature': feat_cols,
    'coefficient': model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
})

coef_df = coef_df.sort_values(
    'odds_ratio',
    ascending=False
)

# =========================================================
# SAVE RESULTS
# =========================================================

output_path = '../outputs/signals/outcome_model_coefs.csv'

coef_df.to_csv(
    output_path,
    index=False
)

print(f"\nModel coefficients saved to:\n{output_path}")

# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\nTOP 10 PREDICTORS OF SERIOUS OUTCOME")
print("=" * 60)

print(
    coef_df.head(10).to_string(index=False)
)

print("\nBOTTOM 10 PREDICTORS")
print("=" * 60)

print(
    coef_df.tail(10).to_string(index=False)
)

Dataset Shape: (528000, 35)

Top Drugs Used:
['TOFACITINIB', 'RISPERIDONE', 'RIVAROXABAN', 'AVANDIA', 'ETANERCEPT', 'DUPILUMAB', 'ADALIMUMAB', 'SODIUM OXYBATE', 'VEDOLIZUMAB', 'PREGABALIN', 'CERTOLIZUMAB PEGOL', 'PALBOCICLIB', 'SECUKINUMAB', 'FINGOLIMOD HCL', 'AMBRISENTAN', 'INFLIXIMAB-DYYB', 'INFLIXIMAB', 'NIVOLUMAB', 'ZANTAC', 'LENALIDOMIDE']

Top Reactions:
['DRUG INEFFECTIVE', 'DEATH', 'PNEUMONIA', 'DRUG HYPERSENSITIVITY', 'MYOCARDIAL INFARCTION', 'GYNAECOMASTIA', 'CHRONIC KIDNEY DISEASE', 'OFF LABEL USE', 'GASTROINTESTINAL HAEMORRHAGE', 'CEREBROVASCULAR ACCIDENT', 'FATIGUE', 'ACUTE KIDNEY INJURY', 'PAIN', 'DIARRHOEA', 'DYSPNOEA', 'NAUSEA', 'MALAISE', 'HEADACHE', 'RENAL INJURY', 'FALL']

Number of Features: 46

Feature Matrix Shape: (528000, 46)

Training Logistic Regression Model...
Model Training Complete!

Model coefficients saved to:
../outputs/signals/outcome_model_coefs.csv

TOP 10 PREDICTORS OF SERIOUS OUTCOME
                         feature  coefficient  odds_ratio
       